# Miras Gezgini: internet fotoğraflarından 3D miras

[MegaScenes](https://megascenes.github.io) veri seti, Wikimedia Commons'taki serbest lisanslı fotoğraflardan binlerce yerin kamera konumlarını hesaplamış. Bu defter seçtiğin UNESCO mirasının fotoğraflarını indirir, 3D Gaussian Splatting ile eğitir ve siteye eklenecek zip'i verir.

**Çalışma zamanı > Çalışma zamanı türünü değiştir > GPU (T4 veya L4)** seçip hücreleri sırayla çalıştır. Sahne başına L4'te ~25 dk, T4'te ~50 dk sürer. Eğitim arka planda çalışır; biten sahneler 5. hücreyle indirilir.

Üretilen sahneler fotoğrafların lisansları gereği **CC BY-SA 4.0** ile paylaşılır; zip içindeki `credits.json` kullanılan Commons dosyalarını listeler.

In [ ]:
#@title 1. GPU kontrolü
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2. Kurulum (8–12 dk)
REPO_URL = "https://github.com/Oguzhan61-dilmac/360-vid2splat.git"  #@param {type:"string"}
import os
if not os.path.exists("/content/360-vid2splat"):
    !git clone -q {REPO_URL} /content/360-vid2splat
%cd /content/360-vid2splat
!git pull -q
!pip install -q numpy opencv-python-headless scipy tqdm ninja
!pip install -q pycolmap==4.2.1  # MegaScenes'te özellik eşleme yok; CPU sürümü yeter
!pip install -q --no-build-isolation gsplat==1.5.3
!node -v || (apt-get -qq update && apt-get -qq install -y nodejs npm)
!gh --version >/dev/null 2>&1 || (curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg -o /usr/share/keyrings/githubcli-archive-keyring.gpg && echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" > /etc/apt/sources.list.d/github-cli.list && apt-get -qq update && apt-get -qq install -y gh)

# gsplat CUDA çekirdeklerini şimdi derle (ilk seferde birkaç dakika sürer)
import torch
from gsplat import rasterization
n = 16
img, _, _ = rasterization(
    torch.randn(n, 3, device="cuda"), torch.randn(n, 4, device="cuda"), torch.rand(n, 3, device="cuda") * 0.1,
    torch.rand(n, device="cuda"), torch.rand(n, 3, device="cuda"),
    torch.tensor([[[1., 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 5], [0, 0, 0, 1]]], device="cuda"),
    torch.tensor([[[100., 0, 50], [0, 100, 50], [0, 0, 1]]], device="cuda"), 100, 100)
import pycolmap
print("Hazır. COLMAP GPU:", pycolmap.has_cuda, "| gsplat çıktı:", tuple(img.shape))

In [ ]:
#@title 3. (İsteğe bağlı) Sahneleri doğrudan GitHub'a yayınlamak için giriş
#@markdown Çıkan kodu https://github.com/login/device adresine gir. İş bitince 7. hücreyle çıkış yap.
import subprocess, time
gh_log = open('/content/gh_login.log', 'w')
gh = subprocess.Popen('gh auth login -h github.com -p https -w', shell=True, stdin=subprocess.PIPE, stdout=gh_log, stderr=subprocess.STDOUT, text=True)
gh.stdin.write('\n'); gh.stdin.flush()
time.sleep(6)
print(open('/content/gh_login.log').read())

In [ ]:
#@title 4. Eğitimi arka planda başlat
#@markdown Virgülle ayır: ayasofya, efes, tac-mahal, sagrada-familia, pantheon, suleymaniye, stonehenge, aziz-vasil, karlov-koprusu, partenon, sfenks, westminster
SAHNELER = "ayasofya,efes,sagrada-familia,pantheon,suleymaniye,stonehenge,aziz-vasil,tac-mahal"  #@param {type:"string"}
ADIM = 20000  #@param {type:"integer"}
YAYINLA = True  #@param {type:"boolean"}
import subprocess
!git -C /content/360-vid2splat pull -q
!gh auth status >/dev/null 2>&1 && gh auth setup-git
log = open('/content/miras_egitim.log', 'a')
proc = subprocess.Popen(f'python -u scripts/megascenes_batch.py --names "{SAHNELER}" --out /content/miras --steps {ADIM}' + (' --publish' if YAYINLA else ''),
                        shell=True, stdout=log, stderr=subprocess.STDOUT, cwd='/content/360-vid2splat')
print('Eğitim arka planda başladı. Durumu 5. hücreyle izle; bağlantı koparsa bu hücreyi yeniden çalıştır, kaldığı yerden devam eder.')

In [ ]:
#@title 5. Durum
!grep -E "=====|seçilen|hizalama|yayınlandı|hazır|HATA|Error" /content/miras_egitim.log | tail -n 14
!cat /content/miras/*/splat/progress.json 2>/dev/null | tr -d '\n '; echo
!ls -la /content/miras/*.zip 2>/dev/null

In [ ]:
#@title 6. Biten zip'leri indir (yayınlamadıysan)
import glob, os
from google.colab import files
done = set(open('/content/indirilen.txt').read().split()) if os.path.exists('/content/indirilen.txt') else set()
for z in sorted(glob.glob('/content/miras/*.zip')):
    if z not in done:
        files.download(z)
        done.add(z)
open('/content/indirilen.txt', 'w').write('\n'.join(sorted(done)))
print(sorted(done))

In [ ]:
#@title 7. GitHub oturumunu kapat
!gh auth logout -h github.com 2>&1 | tail -1
!git config --global --unset-all credential.https://github.com.helper 2>/dev/null; echo kapatıldı